## Import Dataset

In [1]:
# Impot libraries
import pandas as pd

In [2]:
df = pd.read_excel('./datasets/WRMD_2016_to_2024_all_cols.xlsx')
# Using the file above will have more columns that are needed to join on the WCV data
# df = pd.read_csv('WRMD_VA_2016-2025_Records.csv') 

In [3]:
df.columns

Index(['admissions.case_year', 'admissions.hash', 'admissions.id', 'exams.age',
       'exams.age_unit', 'exams.attitude', 'exams.bcs', 'exams.body',
       'exams.cardiopulmonary', 'exams.cns', 'exams.comments',
       'exams.dehydration', 'exams.examined_at', 'exams.examiner',
       'exams.forelimb', 'exams.gastrointestinal', 'exams.head',
       'exams.hindlimb', 'exams.integument', 'exams.mm_color',
       'exams.mm_texture', 'exams.musculoskeletal', 'exams.nutrition',
       'exams.sex', 'exams.temperature', 'exams.temperature_unit',
       'exams.treatment', 'exams.type', 'exams.weight', 'exams.weight_unit',
       'patient_locations.area', 'patient_locations.comments',
       'patient_locations.enclosure', 'patient_locations.moved_in_at',
       'patient_locations.where_holding', 'patients.address_found',
       'patients.admitted_at', 'patients.admitted_by', 'patients.band',
       'patients.carcass_saved', 'patients.care_by_rescuer',
       'patients.city_found', 'patients.cl

In [4]:
df.shape

(25100, 95)

## Drop Rows where there is no geolocation information

In [5]:
# Profile the relevant columns
# df[['patients.address_found', 'patients.county_found', 'patients.city_found']]

In [6]:
#Profile the address_found column
df['patients.address_found'].value_counts()

patients.address_found
ss                                                                                                                                 294
00                                                                                                                                 129
unknown                                                                                                                            124
X                                                                                                                                  120
x                                                                                                                                   91
                                                                                                                                  ... 
776 Old Charles Town Rd.                                                                                                             1
Intersection of Harpers Ferry Rd

Some of the data in 'patients.address_found' is not useful for programatically determining lat long coordinates. We want to drop the rows where the address_found starts with a alphabetic character AND either lat_found or lng_found is null

In [7]:
# Determine How many rows have the 'patients.address_found' column starting with a non-numeric character
df[df['patients.address_found'].str[0].str.isnumeric() == False].shape

(5985, 95)

In [8]:
# Of the rows that have the 'patients.address_found' column starting with a non-numeric character, how many have a null for eithe rlat_found or long_found
df_address_profiling = df[df['patients.address_found'].str[0].str.isnumeric() == False]
df_address_profiling[['patients.lat_found', 'patients.lng_found']].isnull().sum()

patients.lat_found    3965
patients.lng_found    3965
dtype: int64

In [9]:
# determine which rows have the 'patients.address_found' column starting with a non-numeric character 
# df[df['patients.address_found'].str[0].str.isnumeric() == True]

# Set the 'patients.address_found' column to null for these rows
df.loc[df['patients.address_found'].str[0].str.isnumeric() == False, 'patients.address_found'] = None

In [10]:
# Find rows where the 'patients.address_found' column contains only numbers
df[df['patients.address_found'].str.isnumeric() == True]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
1503,2016,NaN,1504,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Reptilia,Emydidae,Terrapene,Testudines,Carolina
1504,2016,NaN,1505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Aves,Mimidae,Mimus,Passeriformes,polyglottos
1505,2016,NaN,1506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Aves,Passeridae,Passer,Passeriformes,domesticus
1506,2016,NaN,1507,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Mammalia,Procyonidae,Procyon,Carnivora,lotor
1507,2016,NaN,1508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Aves,Passeridae,Passer,Passeriformes,domesticus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1842,2017,NaN,202,0.0,Juvenile,Quiet,Reasonable,NaN,NaN,"standing on intake, but weak and quickly sank ...",...,Transported by JoDee Grierber from Pender to B...,NaN,NaN,NaN,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
2280,2017,NaN,640,0.0,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,Wildlife Rescue League,NaN,7034085976,22202,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
2803,2017,NaN,1163,0.0,Juvenile,Alert,Reasonable,NaN,NaN,NaN,...,Former Director of Wildlife Services at BRWC,Wildlife Vet Care,540-664-9494,22646,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus
3027,2017,NaN,1387,0.0,Adult,Alert,Reasonable,NaN,"open-mouth breathing, lungs auscult clear",NaN,...,NaN,Shenandoah Animal Control,NaN,NaN,VA,Aves,Accipitridae,Accipiter,Accipitriformes,cooperii


In [11]:
# For rows where the 'patients.address_found' column contains only numbers, set the 'patients.address_found' column to null
df.loc[df['patients.address_found'].str.isnumeric() == True, 'patients.address_found'] = None

If we have no address information whatsoever, then drop that row:

In [12]:
# Drop rows where all the following columns are empty: 'patients.address_found', 'patients.lat_found', 'patients.long_found'
df = df.dropna(subset=['patients.address_found', 'patients.lat_found', 'patients.lng_found'], how='all')

In [13]:
df.shape

(21109, 95)

In [14]:
# Find how many rows have a null for either the patients.lat_found or patients.lng_found columns
df[['patients.lat_found', 'patients.lng_found']].isnull().sum()

patients.lat_found    15418
patients.lng_found    15417
dtype: int64

Check if we have incomplete coordinates for any of the rows (only one of lat or long):

In [15]:
# Find rows where only one of patients.lat_found or patients.lng_found columns are null
df[df['patients.lat_found'].isnull() ^ df['patients.lng_found'].isnull()]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
7634,2019,NaN,1972,NaN,Infant,Alert,Reasonable,NaN,NaN,NaN,...,NaN,NaN,7033973526,22309,VA,Mammalia,Sciuridae,Sciurus,Rodentia,carolinensis


In [16]:
# If only one of patients.lat_found or patients.lng_found columns are null, set the other to null
df.loc[df['patients.lat_found'].isnull() ^ df['patients.lng_found'].isnull(), ['patients.lat_found', 'patients.lng_found']] = None

In [17]:
# Note that some of the entries in the 'patients.address_found' column are not valid addresses.
# Search for the word "mile" within the string to see examples of these entries. Roughly 50

In [18]:
df['patients.address_found'].value_counts()

patients.address_found
106 Island Farm Ln         75
106 Island Farm Lane       65
7503 Cedar Knolls Drive    18
32 Jenkins Ct              16
3307 Halfway Rd            16
                           ..
45871 Debhill Terrace       1
67 Cedar Mountain           1
13115 Hill Club Ln          1
9 Rosepetal St              1
415 5th St                  1
Name: count, Length: 13640, dtype: int64

## Determine Which ones are Vehicle Collision Related

* The patients.keyword column contains relevant information. Search for "HBV" (Hit by Vehicle).
* The patients.diagnosis column also has relevant data - "HBV" and "HBV" as a string in long from text
* the patient.reasons_for_admission column has relevent info: "HBV" among others


### Keyword Column

In [19]:
# Find all rows in the patient.keywords column that contain the string "HBV" and examine the results
vc = df[df['patients.keywords'].str.contains('HBV', regex=False, case=False, na=False)]['patients.keywords'].value_counts()
print(vc)

patients.keywords
suspect HBV                                                536
B - Baby, orphan, mother HBV                               122
B - Baby, mother HBV                                        72
B - Baby, mother HBV, orphan                                43
B - Baby, small, mother HBV                                 24
                                                          ... 
suspect HBV,  suspect cat attack                             1
gunshot, suspect HBV                                         1
suspect HBV, rabies suspect                                  1
B - Baby, mother killed, dog attack, mother HBV, orphan      1
grounded, trauma, suspect HBV                                1
Name: count, Length: 76, dtype: int64


In [20]:
# Grab the indexes of the rows that contain the string "HBV" in the patient.keywords column
keyword_column_hbv = df[df['patients.keywords'].str.contains('HBV', regex=False, case=False, na=False)].index

### Diagnosis Column

In [21]:
df['patients.diagnosis'].value_counts()

patients.diagnosis
T - Unknown Trauma            4017
B - Baby                      3588
D - Domestic animal attack    3542
M - Human non-intentional     1224
H - Hit by Vehicle            1171
                              ... 
H - hit by vehicle               1
T - Unknown traum                1
predator attack                  1
t                                1
B - Baby, B - Baby               1
Name: count, Length: 113, dtype: int64

In [22]:
# Find all rows in the patient.diagnosis column that are equal to "H - Hit by Vehicle" and examine the results
df[df['patients.diagnosis'].str.contains('Vehicle', regex=False, case=False, na=False)]['patients.diagnosis'].value_counts()

patients.diagnosis
H - Hit by Vehicle     1171
H - Hit by vehicle      233
H- Hit by vehicle        84
H - Hit by vehicle       39
H- hit by vehicle         1
H -Hit by vehicle         1
H - hit by vehicle        1
Name: count, dtype: int64

In [23]:
# Grab the indexes of the rows that contain the string "Vehicle" in the patient.diagnosis column
diagnosis_column_vehicle = df[df['patients.diagnosis'].str.contains('Vehicle', regex=False, case=False, na=False)].index


### Reasons for Admission Column

In [24]:
value_counts_reasons_for_admission = df['patients.reasons_for_admission'].value_counts()
print(value_counts_reasons_for_admission)

patients.reasons_for_admission
cat attack                                             609
Cat attack                                             433
ss                                                     277
HBV                                                    259
dog attack                                             229
                                                      ... 
found on playground alone.  able to contain.  fleas      1
hatchling stepped on by toddlers                         1
on road, crow starting to attack                         1
cat caught eyes closed infant                            1
unable to fly.  no obv inj.  Otherwise BAR.              1
Name: count, Length: 12250, dtype: int64


In [25]:
# Set up a regex pattern to match the string "vehicle" or "HBV" or "collision" in the patient.reasons_for_admission column
# vehicle_collision_regex = "vehicle|HBV|HBC|collision|car"
vehicle_collision_regex = r"\bvehicle\b|\bHBV\b|\bHBC\b|\bcollision\b|\bcar\b"
# Find rows where the patients.ressons_for_admission column contains the string "vehicle" or "HBV" or "collision" and examine the results
df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
0,2016,NaN,1,0.0,Adult,Depressed,Good,NaN,congested breathing,"unable to stay sternal, ataxia",...,--,--,508-360-2061,22931,VA,Mammalia,Didelphidae,Didelphis,Didelphimorphia,virginiana
38,2016,NaN,39,0.0,Adult,Quiet,Thin,NaN,NaN,NaN,...,NaN,NaN,5712687699,25419,WA,Aves,Ardeidae,Ardea,Pelecaniformes,herodias
43,2016,NaN,44,0.0,Adult,Alert,Reasonable,NaN,NaN,"not standing on R foot normally, falling over ...",...,NaN,--,5402224303,20186,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus
53,2016,NaN,54,0.0,Adult,Depressed,Thin,NaN,NaN,"nystagmus, not using rear legs",...,NaN,--,3048864197,25430,WV,Mammalia,Leporidae,Sylvilagus,Lagomorpha,floridanus
55,2016,NaN,56,0.0,Adult,Stuporous,Thin,NaN,NaN,NaN,...,found by Troy: 571-510-2595,--,2032432010,20135,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24996,2024,NaN,3793,NaN,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,NaN,NaN,2672661277,18360,PA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25024,2024,NaN,3821,NaN,Adult,Alert,Reasonable,NaN,NaN,weak leg use,...,NaN,NaN,5408776642,22602,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25029,2024,NaN,3826,NaN,Adult,Obtunded,Reasonable,NaN,NaN,"obtunded and sternal but legs work, no sign of...",...,NaN,NaN,5402701905,20186,VA,Mammalia,Leporidae,Sylvilagus,Lagomorpha,floridanus
25041,2024,NaN,3838,NaN,Adult,Obtunded,Reasonable,NaN,"moderate dyspnea, open mouthed and tachypneic/...",not standing - weak legs but can kick/grip wit...,...,NaN,NaN,7033031988,22407,VA,Aves,Cardinalidae,Cardinalis,Passeriformes,cardinalis


In [26]:
# df_reasons_for_admission = df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)]
# reasons_for_admission_vc = df_reasons_for_admission['patients.reasons_for_admission'].value_counts()
# print(reasons_for_admission_vc)

In [27]:
# test_string = "Stray cat attacked bird in yard.  Was able to get bird + placed in carrier"
# # Check if the string above matches the regex pattern: "vehicle|HBV|HBC|collision|car"
# import re
# # re.search(vehicle_collision_regex, test_string, re.IGNORECASE)
# # We don't want to match the word "carrier" in the string above with the regex pattern.
# # We can use word boundaries to match the whole word "car" instead of just the substring "car" in the string above
# vehicle_collision_regex = r"\bvehicle\b|\bHBV\b|\bHBC\b|\bcollision\b|\bcar\b"
# re.search(vehicle_collision_regex, test_string, re.IGNORECASE)

In [28]:
# Grab the indexes of the rows that contain the string "Vehicle" in the patient.diagnosis column
reasonforadmission_column_vehicle = df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)].index

### Unify hit by vechicle indexes

In [29]:
# keyword_column_hbv

In [30]:
# Unify the indexes of the rows that we suspect are related to vehicle collisions
unified_indexes = keyword_column_hbv.union(diagnosis_column_vehicle).union(reasonforadmission_column_vehicle)

In [31]:
# Create a new DataFrame with only the rows that we suspect are related to vehicle collisions
df_vehicle_collisions = df.loc[unified_indexes]

In [32]:
df_vehicle_collisions.shape

(3049, 95)

### Examine disposition lat and long columns

Examining the disposition_lat and disposition_lng columns often gives the coordination of wildlife center or animal hospitals. Suspect that this column is not meaningful for our goal of identifying hot spots of animal-vehicle conflict

In [33]:
df_examime_location = df[['patients.disposition_lat',
       'patients.disposition_lng', 'patients.disposition_location',
       'patients.disposition_subdivision', 'patients.dispositioned_at',
       'patients.dispositioned_by', 'patients.found_at', 'patients.keywords',
       'patients.lat_found', 'patients.lng_found']]

In [34]:
# select rows from df_examiem_location_cols where patients.disposition_lat is not null
df_examime_location[df_examime_location['patients.disposition_lat'].notnull()]

,patients.disposition_lat,patients.disposition_lng,patients.disposition_location,patients.disposition_subdivision,patients.dispositioned_at,patients.dispositioned_by,patients.found_at,patients.keywords,patients.lat_found,patients.lng_found
0,37.522251,-78.668194,--,VA,2016-01-02,JB,2016-01-01,NaN,38.960822,-77.741824
2,-78.656894,37.431573,NaN,VA,2016-01-07,JA,2016-01-06,NaN,-77.738882,39.325379
3,37.522251,-78.668194,NaN,VA,2016-01-17,JA,2016-01-08,suspected predator attack,-78.157186,39.184540
7,-78.656894,37.431573,found location,VA,2016-04-23,HS,2016-01-10,NaN,38.822899,-78.746116
10,37.601787,-80.823972,Buck Hill Rd,WV,2016-03-10,finder,2016-01-15,NaN,39.400742,-78.115627
...,...,...,...,...,...,...,...,...,...,...
24979,39.048614,-78.060132,Boyce,VA,2024-11-25,SL,2024-11-25,"B - Baby, orphan",NaN,NaN
25016,39.169668,-78.168560,Winchester,VA,2024-12-08,JR,2024-11-30,"B - Baby, renesting not possible, nest destroyed",38.625541,-77.271644
25017,39.169668,-78.168560,Winchester,VA,2024-12-08,JR,2024-11-30,"B - Baby, renesting not possible, nest destroyed",38.625541,-77.271644
25018,39.169668,-78.168560,Winchester,VA,2024-12-08,JR,2024-11-30,"B - Baby, renesting not possible, nest destroyed",38.625541,-77.271644


## Convert Address to Geolocation
For rows where we have an address but no lat or long info

### Filter out rows that already have a lat / long entry

In [35]:
# Find rows where both patients.lat_found and patients.lng_found columns are null
df_geocoding_req = df_vehicle_collisions[df_vehicle_collisions['patients.lat_found'].isnull() & df_vehicle_collisions['patients.lng_found'].isnull()]

In [36]:
df_geocoding_req

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
33,2016,NaN,34,0.0,Adult,Alert,Reasonable,NaN,breathing rapid/shallow,"unable to stand, but can grip a little with feet",...,willing to p/u and release,--,540-662-0143,22656,VA,Aves,Cardinalidae,Cardinalis,Passeriformes,cardinalis
618,2016,NaN,619,NaN,Adult,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5402332523,22025,VA,Mammalia,Sciuridae,Marmota,Rodentia,monax
942,2016,NaN,943,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Reptilia,Emydidae,Terrapene,Testudines,Carolina
1424,2016,NaN,1425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,VA,Mammalia,Sciuridae,Sciurus,Rodentia,carolinensis
1674,2017,NaN,34,0.0,Adult,Alert,Thin,NaN,NaN,NaN,...,NaN,NaN,540-687-5653,NaN,VA,Aves,Strigidae,Megascops,Strigiformes,asio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25029,2024,NaN,3826,NaN,Adult,Obtunded,Reasonable,NaN,NaN,"obtunded and sternal but legs work, no sign of...",...,NaN,NaN,5402701905,20186,VA,Mammalia,Leporidae,Sylvilagus,Lagomorpha,floridanus
25041,2024,NaN,3838,NaN,Adult,Obtunded,Reasonable,NaN,"moderate dyspnea, open mouthed and tachypneic/...",not standing - weak legs but can kick/grip wit...,...,NaN,NaN,7033031988,22407,VA,Aves,Cardinalidae,Cardinalis,Passeriformes,cardinalis
25047,2024,NaN,3844,NaN,Adult,Quiet,Reasonable,severlely displaced open comminuted L humeral ...,NaN,NaN,...,NaN,NaN,5403168052,20135,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus
25061,2024,NaN,3858,NaN,Adult,Obtunded,Good,NaN,NaN,"see eent, otherwise just generalized weakness,...",...,Finder Julie brought the owl to BRVA 12/20 @7pm,Blue Ridge Vet Associates,5404540035,20132,VA,Aves,Strigidae,Strix,Strigiformes,varia


### Export the data frame that needs GeoCoding

In [37]:
df_geocoding_req.to_pickle('./datasets/WRMD_2014_to_2025_geocoding_req.pkl')

In [38]:
from geopy.geocoders import GoogleV3
from geopy.exc import GeocoderTimedOut
import time
import os
from concurrent.futures import ThreadPoolExecutor


In [ ]:
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
geolocator = GoogleV3(api_key=GOOGLE_API_KEY)

In [ ]:
import pandas as pd
from geopy.geocoders import GoogleV3
from geopy.exc import GeocoderTimedOut
import time
from concurrent.futures import ThreadPoolExecutor

# Initialize geolocator with a custom timeout value (in seconds)
geolocator = Nominatim(user_agent="Geopy Library")

# Cache to store geocoded addresses (address as key, (lat, lng) as value) to avoid repeating 
cache = {}

# Function to geocode an address 
def geocode_address(address, retries=3, delay=2):
    """
    Geocode an address with retry logic and timeout handling.
    Includes caching to avoid re-geocoding the same address.
    
    param address: The address to geocode.
    param retries: Number of retries on timeout (default 3).
    param delay: Delay (in seconds) between retries (default 2).
    return: Tuple (latitude, longitude) or (None, None) if not found.
    """
    if address in cache:
        return cache[address]  # Return cached result
    
    for attempt in range(retries):
        try:
            # Attempt to geocode with a 5-second timeout
            location = geolocator.geocode(address, timeout=5)
            
            if location:
                # Cache the result for future use
                cache[address] = (location.latitude, location.longitude)
                return location.latitude, location.longitude
            else:
                return None, None
        
        except GeocoderTimedOut:
            time.sleep(delay)  # Sleep before retrying
        except Exception as e:
            time.sleep(delay)  # Sleep before retrying
    
    return None, None

# Function to process each row in the DataFrame and update coordinates
def process_row(index, row):
    address = row['patients.address_found']
    if pd.notna(address):  # Only geocode if address is not NaN
        lat, lng = geocode_address(address)
        return index, lat, lng
    return index, None, None

# Function to populate missing latitude and longitude in the DataFrame using multi-threading
def populate_missing_coordinates(df):
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(lambda row: process_row(*row), df.iterrows()))
        
    for index, lat, lng in results:
        if lat is not None and lng is not None:
            df.at[index, 'patients.lat_found'] = lat
            df.at[index, 'patients.lng_found'] = lng
    return df

# Call the function to update missing latitude and longitude
df = populate_missing_coordinates(df)

# Output the updated DataFrame
df


## Handle Rows that don't need Geocoding
These rows already have a lat long provided, but need further cleanup since some of the locations are unreasonable

* [x] Some rows have lat long swapped
* [x] Some rows have the decimal point in the wrong position
* [] Some that are not anywhere near VA -WIP
* Some 

In [39]:
# Get the rows that have both latitude and longitude
df_geocoding_not_req = df_vehicle_collisions[df_vehicle_collisions['patients.lat_found'].notnull() & df_vehicle_collisions['patients.lng_found'].notnull()]

In [40]:
df_geocoding_not_req

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
0,2016,NaN,1,0.0,Adult,Depressed,Good,NaN,congested breathing,"unable to stay sternal, ataxia",...,--,--,508-360-2061,22931,VA,Mammalia,Didelphidae,Didelphis,Didelphimorphia,virginiana
2,2016,NaN,3,0.0,Adult,Alert,Emaciated,NaN,NaN,NaN,...,brought in by Erin,--,304-876-1068,--,OH,Aves,Cathartidae,Coragyps,Cathartiformes,atratus
8,2016,NaN,9,0.0,Adult,Depressed,Thin,NaN,NaN,barely responsive,...,--,--,5403032556,22603,VA,Aves,Strigidae,Strix,Strigiformes,varia
10,2016,NaN,11,0.0,Adult,Quiet,Thin,NaN,NaN,NaN,...,willing to release when ready,Frederick County Landfill,3046715230,--,VA,Aves,Strigidae,Strix,Strigiformes,varia
16,2016,NaN,17,0.0,Adult,Quiet,Reasonable,NaN,rapid and shallow breathing,NaN,...,--,--,7037370110,20176,VA,Aves,Turdidae,Turdus,Passeriformes,migratorius
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24955,2024,NaN,3752,NaN,Adult,Obtunded,Good,NaN,"moderately increased respiratory effort, open ...",laterally recumbent in finder's backseat,...,NaN,NaN,9154870726,25430,WV,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
24996,2024,NaN,3793,NaN,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,NaN,NaN,2672661277,18360,PA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25024,2024,NaN,3821,NaN,Adult,Alert,Reasonable,NaN,NaN,weak leg use,...,NaN,NaN,5408776642,22602,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25049,2024,NaN,3846,NaN,Juvenile,Quiet,Plump,NaN,NaN,NaN,...,NaN,Stafford County Animal Control,NaN,NaN,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus


### Handling Lat and Long that are Impossible
Upon examination of the lat and long values that are impossible, it appears that the decimal point is simply misplaced. We correct that here.

In [41]:
# Identify the rows that have a latitude or longitude that is too large to be valid
lat_or_long_too_large = df_geocoding_not_req[(df_geocoding_not_req['patients.lat_found'] > 90) | (df_geocoding_not_req['patients.lat_found'] < -90) | (df_geocoding_not_req['patients.lng_found'] > 180) | (df_geocoding_not_req['patients.lng_found'] < -180)]

In [42]:
# Count and display the number of rows with latitude or longitude that are outside the possible range
lat_or_long_too_large[['patients.lng_found', 'patients.lat_found']]

,patients.lng_found,patients.lat_found
13492,-78033331.0,38.568800
17279,-77865673.0,38.924758
17414,-77531010.0,38.694630
17521,-78150519.0,39.189661
17739,-78096620.0,38.957299
18272,-77703743.0,38.147664
18431,-78150996.0,38.375340
19204,-78276342.0,38.854925
19415,-77820310.0,38.768971
20310,-78375713.0,38.968420


In [43]:
def fix_coordinate_decimal(coord, coord_type='lng'):
    """
    Fix coordinate values that have misplaced decimal points.
    
    Args:
        coord: The coordinate value to fix (latitude or longitude)
        coord_type: 'lat' for latitude (-90 to 90) or 'lng' for longitude (-180 to 180)
      
      Returns:
          Fixed coordinate value within valid range
    """
    if pd.isna(coord):
        return coord

    # Define valid ranges
    if coord_type == 'lat':
        min_val, max_val = -90, 90
    else:  # longitude
        min_val, max_val = -180, 180

    # If already in valid range, return as-is
    if min_val <= coord <= max_val:
        return coord

    # Try different decimal adjustments
    original = coord

    # For very large values, try dividing by powers of 10
    for divisor in [10000, 1000, 100, 10]:
        adjusted = coord / divisor
        if min_val <= adjusted <= max_val:
            return adjusted

    # For values that are still too large, try more aggressive division
    for divisor in [100000, 1000000]:
        adjusted = coord / divisor
        if min_val <= adjusted <= max_val:
            return adjusted

    # If no adjustment works, return the original value
    print(f"Warning: Coordinate {original} is out of valid range for {coord_type}. Returning original value.")
    return coord

In [44]:
df_geocoding_not_req = df_geocoding_not_req.copy()
df_geocoding_not_req['patients.lng_found'] = df_geocoding_not_req['patients.lng_found'].apply(lambda x: fix_coordinate_decimal(x, 'lng'))
df_geocoding_not_req['patients.lat_found'] = df_geocoding_not_req['patients.lat_found'].apply(lambda x: fix_coordinate_decimal(x, 'lat'))

In [45]:
# Assert that all latitude and longitude values are now within valid ranges
assert df_geocoding_not_req['patients.lat_found'].between(-90, 90).all(), "Some latitude values are still out of range!"
assert df_geocoding_not_req['patients.lng_found'].between(-180, 180).all(), "Some longitude values are still out of range!"

### Handle Rows where Lat and Long are Swapped

In [46]:
# Identify rows where latitude and longitude are swapped
region_bounds = {
    'virginia': {
    'lat_min': 36.5,
    'lat_max': 39.5, 
    'lng_min': -83.5,
    'lng_max': -75.5 
    },
    'mid_atlantic': {
        'lat_min': 35.0,
        'lat_max': 42.0,    #Broader Mid-Atlantic
        'lng_min': -85.0,
        'lng_max': -74.0
    },
}

def check_lat_long_swapped(lat, lng, region='mid_atlantic'):
    bounds = region_bounds[region]
    lat_should_be_positive: bool = bounds['lat_min'] > 0
    lng_should_be_negative: bool = bounds['lng_max'] < 0
    lat_swapped: bool = False
    lng_swapped: bool = False
    if lat_should_be_positive and lat < 0:
        lat_swapped = True
    if lng_should_be_negative and lng > 0:
        lng_swapped = True
    both_lat_long_swapped: bool = lat_swapped and lng_swapped
    return both_lat_long_swapped

In [47]:
# Return the rows where the check_lat_long_swapped function returns True
swapped_lat_long = df_geocoding_not_req[df_geocoding_not_req.apply(lambda row: check_lat_long_swapped(row['patients.lat_found'], row['patients.lng_found']), axis=1)]

In [48]:
swapped_lat_long[['patients.lat_found', 'patients.lng_found']]

,patients.lat_found,patients.lng_found
2,-77.738882,39.325379
36,-73.179832,43.610622
37,-78.144800,39.181479
38,-77.577312,38.270378
66,-78.338584,39.204706
...,...,...
4987,-78.092777,39.119129
4988,-77.466481,39.008977
4989,-77.540322,39.116214
4992,-77.832863,38.638918


In [49]:
# Swap the latitude and longitude for the swapped_lat_long DataFrame
def swap_lat_long(lat, lng):
    return lng, lat
swapped_lat_long[['patients.lat_found', 'patients.lng_found']] = swapped_lat_long.apply(lambda row: swap_lat_long(row['patients.lat_found'], row['patients.lng_found']), axis=1, result_type='expand')

/tmp/ipykernel_432956/1235914990.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  swapped_lat_long[['patients.lat_found', 'patients.lng_found']] = swapped_lat_long.apply(lambda row: swap_lat_long(row['patients.lat_found'], row['patients.lng_found']), axis=1, result_type='expand')


In [50]:
swapped_lat_long[['patients.lat_found', 'patients.lng_found']]

,patients.lat_found,patients.lng_found
2,39.325379,-77.738882
36,43.610622,-73.179832
37,39.181479,-78.144800
38,38.270378,-77.577312
66,39.204706,-78.338584
...,...,...
4987,39.119129,-78.092777
4988,39.008977,-77.466481
4989,39.116214,-77.540322
4992,38.638918,-77.832863


In [51]:
# based on the index in the swapped_lat_long DataFrame, update the corresponding rows in the df_geocoding_not_req DataFrame
df_geocoding_not_req.update(swapped_lat_long)

In [52]:
df_geocoding_not_req

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
0,2016,NaN,1,0.0,Adult,Depressed,Good,NaN,congested breathing,"unable to stay sternal, ataxia",...,--,--,508-360-2061,22931,VA,Mammalia,Didelphidae,Didelphis,Didelphimorphia,virginiana
2,2016,NaN,3,0.0,Adult,Alert,Emaciated,NaN,NaN,NaN,...,brought in by Erin,--,304-876-1068,--,OH,Aves,Cathartidae,Coragyps,Cathartiformes,atratus
8,2016,NaN,9,0.0,Adult,Depressed,Thin,NaN,NaN,barely responsive,...,--,--,5403032556,22603,VA,Aves,Strigidae,Strix,Strigiformes,varia
10,2016,NaN,11,0.0,Adult,Quiet,Thin,NaN,NaN,NaN,...,willing to release when ready,Frederick County Landfill,3046715230,--,VA,Aves,Strigidae,Strix,Strigiformes,varia
16,2016,NaN,17,0.0,Adult,Quiet,Reasonable,NaN,rapid and shallow breathing,NaN,...,--,--,7037370110,20176,VA,Aves,Turdidae,Turdus,Passeriformes,migratorius
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24955,2024,NaN,3752,NaN,Adult,Obtunded,Good,NaN,"moderately increased respiratory effort, open ...",laterally recumbent in finder's backseat,...,NaN,NaN,9154870726,25430,WV,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
24996,2024,NaN,3793,NaN,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,NaN,NaN,2672661277,18360,PA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25024,2024,NaN,3821,NaN,Adult,Alert,Reasonable,NaN,NaN,weak leg use,...,NaN,NaN,5408776642,22602,VA,Aves,Accipitridae,Buteo,Accipitriformes,jamaicensis
25049,2024,NaN,3846,NaN,Juvenile,Quiet,Plump,NaN,NaN,NaN,...,NaN,Stafford County Animal Control,NaN,NaN,VA,Aves,Accipitridae,Buteo,Accipitriformes,lineatus


### Identify and Fix rows where the lng is not negative

In [53]:
#Identify rows where the lng_found is not negative
df_geocoding_not_req[df_geocoding_not_req['patients.lng_found'] > 0]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
1633,2016,NaN,1634,NaN,Adult,Quiet,NaN,fleas,NaN,nystagmus,...,NaN,NaN,3052401956,01007,MA,Mammalia,Leporidae,Sylvilagus,Lagomorpha,floridanus
10703,2020,NaN,2759,NaN,Adult,Quiet,Reasonable,NaN,NaN,NaN,...,NaN,NaN,2566170985,22652,VA,Aves,Strigidae,Megascops,Strigiformes,asio
12189,2021,NaN,1379,NaN,Adult,Depressed,Reasonable,"R caudal carapace fracture, deep with maggots ...",NaN,NaN,...,NaN,NaN,5404283917,20186,VA,Reptilia,Emydidae,Terrapene,Testudines,Carolina
13814,2021,NaN,3004,NaN,Adult,Stuporous,Reasonable,NaN,NaN,stuporous,...,**Permission to release at Eastern View High S...,NaN,8458264184,22960,VA,Reptilia,Chelydridae,Chelydra,Testudines,Serpentina
15100,2022,NaN,958,NaN,Adult,Alert,Reasonable,NaN,NaN,NaN,...,NaN,NaN,3044334036,25425,WV,Reptilia,Emydidae,Trachemys,Testudines,Scripta
17765,2023,NaN,214,NaN,Adult,Non-responsive,Reasonable,"no hindlimb deep pain, started agonal breathin...",NaN,NaN,...,NaN,NaN,NaN,20176,VA,Aves,Cathartidae,Coragyps,Cathartiformes,atratus
21153,2023,NaN,3600,NaN,Adult,Obtunded,Reasonable,NaN,NaN,obtunded,...,NaN,NaN,5403255482,22657,VA,Mammalia,Procyonidae,Procyon,Carnivora,lotor
21597,2024,NaN,396,NaN,Adult,Alert,Thin,NaN,visible and open to coelom on underside of cra...,NaN,...,"Willing to release, if available",NaN,7032442069,22630,VA,Reptilia,Emydidae,Chrysemys,Testudines,Picta
21916,2024,NaN,714,NaN,Sub-adult,Non-responsive,Thin,NaN,NaN,Laterally recumbent on presentation. Exhibited...,...,NaN,NaN,5403366255,22655,VA,Mammalia,Procyonidae,Procyon,Carnivora,lotor
21964,2024,NaN,762,NaN,Adult,Non-responsive,Thin,NaN,"Heart externalized through wall, not beating",NaN,...,NaN,NaN,7034756760,22611,VA,Reptilia,Colubridae,Pantherophis,Squamata,alleghaniensis


In [54]:
# Turn all the longitude values negative
df_geocoding_not_req['patients.lng_found'] = df_geocoding_not_req['patients.lng_found'].apply(lambda x: -abs(x))

In [55]:
assert df_geocoding_not_req[df_geocoding_not_req['patients.lng_found'] > 0].shape[0] == 0

### Identify points outside of VA

In [67]:
from shapely.geometry import Point
import geopandas as gpd

shape_file = "./datasets/cb_2018_us_state_20m/cb_2018_us_state_20m.shp"
all_states = gpd.read_file(shape_file)
virginia = all_states[all_states['STUSPS'] == 'VA']
print("✅ Loaded Virginia shapefile")
print("Virginia CRS:", virginia.crs)

virginia_utm = virginia.to_crs('EPSG:32617')
virginia_buffered_geometry = virginia_utm.geometry.buffer(40233.6)  # 25 miles = 40,233.6 meters
virginia_buffered_utm = gpd.GeoDataFrame(virginia_utm, geometry=virginia_buffered_geometry)
virginia_buffered = virginia_buffered_utm.to_crs('EPSG:4269')  # Back to original CRS

geometry = [Point(xy) for xy in zip(df_geocoding_not_req['patients.lng_found'], df_geocoding_not_req['patients.lat_found'])]
points_gdf = gpd.GeoDataFrame(df_geocoding_not_req, geometry=geometry)

# Set CRS to WGS84 (EPSG:4326) if not already set
if points_gdf.crs is None:
    points_gdf.set_crs(epsg=4326, inplace=True)
    print("✅ Set CRS for points to EPSG:4326 (WGS84)")

print("Points CRS:", points_gdf.crs)

# Check and align CRS 
if points_gdf.crs != virginia_buffered.crs:
    print("⚠️ CRS mismatch detected. Reprojecting points to match Virginia shapefile...")
    points_gdf = points_gdf.to_crs(virginia_buffered.crs)
    print("Points reprojected. CRS is now:", points_gdf.crs)
else:
    print("CRS match. No reprojection needed.")

#  Spatial join to find which points are within Virginia
joined = gpd.sjoin(points_gdf, virginia_buffered, how="left", predicate='within')

# Points outside Virginia will have NaN in 'index_right' (no match found)
outside_va = joined[joined['index_right'].isna()]

✅ Loaded Virginia shapefile
Virginia CRS: EPSG:4269
✅ Set CRS for points to EPSG:4326 (WGS84)
Points CRS: EPSG:4326
⚠️ CRS mismatch detected. Reprojecting points to match Virginia shapefile...
Points reprojected. CRS is now: EPSG:4269


In [71]:
# Select the rows that are inside va
inside_va = joined[joined['index_right'].notna()]

In [72]:
import folium
from folium import plugins

# Create base map centered on Virginia
m = folium.Map(location=[37.5, -78.5],
zoom_start=7)

folium.plugins.MeasureControl(primary_length_unit='kilometers').add_to(m)

# Add the "outside" points with popups showing coordinates
for idx, row in outside_va.iterrows():
    folium.Marker(
        [row.geometry.y, row.geometry.x],
        popup=f"Lat: {row.geometry.y:.6f}, Lng: {row.geometry.x:.6f}",
        icon=folium.Icon(color='red')
    ).add_to(m)

m

### Export rows that don't need geocoding
After fixing the outstanding Geocoding issues

In [73]:

# df_geocoding_not_req.to_pickle('./datasets/WRMD_2014_to_2025_geocoding_not_req.pkl')
df_geocoding_not_req.to_pickle('./datasets/WRMD_2016_to_2024_all_cols_geocoding_not_req.pkl')

In [74]:
inside_va

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,index_right,STATEFP,STATENS,AFFGEOID,GEOID,STUSPS,NAME,LSAD,ALAND,AWATER
0,2016,NaN,1,0.0,Adult,Depressed,Good,NaN,congested breathing,"unable to stay sternal, ataxia",...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
2,2016,NaN,3,0.0,Adult,Alert,Emaciated,NaN,NaN,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
8,2016,NaN,9,0.0,Adult,Depressed,Thin,NaN,NaN,barely responsive,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
10,2016,NaN,11,0.0,Adult,Quiet,Thin,NaN,NaN,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
16,2016,NaN,17,0.0,Adult,Quiet,Reasonable,NaN,rapid and shallow breathing,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24955,2024,NaN,3752,NaN,Adult,Obtunded,Good,NaN,"moderately increased respiratory effort, open ...",laterally recumbent in finder's backseat,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
24996,2024,NaN,3793,NaN,Juvenile,Quiet,Reasonable,NaN,NaN,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
25024,2024,NaN,3821,NaN,Adult,Alert,Reasonable,NaN,NaN,weak leg use,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
25049,2024,NaN,3846,NaN,Juvenile,Quiet,Plump,NaN,NaN,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09


In [75]:
# Write the inside_va DataFrame to a pickle file
inside_va.to_pickle('./datasets/WRMD_2016_to_2024_inside_va_geocoding_not_required.pkl')